In [28]:
import spacy, textstat, pandas as pd, numpy as np, json, os, re
nlp = spacy.load("en_core_web_sm")

In [16]:
with open("llm_explanations/xaiqb_human_verbose.json") as f:
    xaiqb_human = json.load(f)

llm_responses = {}
for folder in ["run_1", "run_2", "run_3"]:
    llm_responses.update({folder: {}})
    for file in [x for x in os.listdir(f"llm_explanations/{folder}") if re.search(".json", x) is not None]:
        with open(f"llm_explanations/{folder}/{file}") as f:
                llm_responses[folder].update({file.replace(".json", ""): json.load(f)})

In [75]:
text_analysis = llm_responses["run_1"]["xaiqb_gemini_3_pro"]
text_analysis_df = []


for model, responses in zip(["ChatGPT", "Claude", "Gemini", "Human"], 
                            [llm_responses["run_2"]["xaiqb_chatgpt_5.2_extended_thinking"], 
                             llm_responses["run_3"]["xaiqb_claude_opus_4.6_reasoning"], 
                             llm_responses["run_1"]["xaiqb_gemini_3_pro"],
                             xaiqb_human]):
        for category in responses["questions"].keys():
                for question in responses["questions"][category].keys():

                        answer = responses["questions"][category][question]["answer"]
                        answer_nlp = nlp(responses["questions"][category][question]["answer"])

                        text_analysis_df += [pd.DataFrame({
                                "Model": model,
                                "Category": category,
                                "Question": question,
                                "Answer": responses["questions"][category][question]["answer"],
                                "Sentence count": [len(list(answer_nlp.sents))],
                                "Word count": [len([t for t in answer_nlp if not t.is_punct])],
                                "Difficult words": [sum([textstat.syllable_count(str(t)) >= 3 for t in answer_nlp if not t.is_punct])],
                                }).assign(**({"Word / sentence": lambda x: x["Word count"] / x["Sentence count"]}))]

text_analysis_df = pd.concat(text_analysis_df)
text_analysis_df.head()

,Model,Category,Question,Answer,Sentence count,Word count,Difficult words,Word / sentence
0,ChatGPT,Data,What kind of data was the system trained on?,The system was trained on responses from a hou...,3,74,14,24.666667
0,ChatGPT,Data,What is the source of the training data?,The training data come from the National 2009 ...,3,72,16,24.000000
0,ChatGPT,Data,How were the labels/ground-truth produced?,Ground truth labels are binary indicators reco...,3,51,9,17.000000
0,ChatGPT,Data,What is the sample size of the training data?,"There are 26,707 respondents in the combined t...",3,39,4,13.000000
0,ChatGPT,Data,What dataset(s) is the system NOT using?,The system does not use the respondent identif...,3,46,14,15.333333


In [80]:
text_analysis_df.groupby(["Model"]).agg({x: "mean" for x in ["Sentence count", "Word count", "Word / sentence", "Difficult words"]}).reset_index().sort_values(["Model"]).T.to_clipboard()